# PDF to Markdown extraction


In [ ]:
import base64
import os
import re
from pathlib import Path

from dotenv import load_dotenv
from mistralai.client import Mistral

PROJECT_ROOT = Path("..").resolve()
load_dotenv(PROJECT_ROOT / ".env")
PDF_PATH = PROJECT_ROOT / "data/class-10/textbooks/jemh101.pdf"
BOOK_ID = "jemh101"
OUTPUT_DIR = PROJECT_ROOT / "extracted_data/class_10" / BOOK_ID

client = Mistral(api_key=os.getenv("MISTRAL_API_KEY"))

In [ ]:
IMAGE_DESCRIPTION_PROMPT = """\
You are describing figures from a school textbook.

Describe this image in detail. Return markdown with these sections:
- **Figure Type** (diagram, graph, photo, illustration, etc.)
- **Description**
- **Text in Image** (transcribe any visible text)
- **Key Information** (labels, values, relationships)

If it is a graph, explain the axes and trends.
If it is a diagram, explain the relationships between parts.
If it is a table, describe the rows and columns.

Do not hallucinate. Only describe what is visible."""


def describe_image(client: Mistral, image_base64: str) -> str:
    """Use Pixtral to turn an image into a markdown description."""
    response = client.chat.complete(
        model="pixtral-12b-latest",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": IMAGE_DESCRIPTION_PROMPT},
                    {"type": "image_url", "image_url": image_base64},
                ],
            }
        ],
    )
    return response.choices[0].message.content


def save_extraction(ocr_response, output_dir: Path, book_id: str, client: Mistral) -> Path:
    """Write OCR output with separate tables/ and image-description .md files."""
    output_dir = Path(output_dir)
    tables_dir = output_dir / "tables"
    images_dir = output_dir / "images"
    tables_dir.mkdir(parents=True, exist_ok=True)
    images_dir.mkdir(parents=True, exist_ok=True)

    # Remove any previously saved binary image files
    for old_image in images_dir.glob("*"):
        if old_image.suffix.lower() in {".jpeg", ".jpg", ".png", ".gif", ".webp"}:
            old_image.unlink()

    page_markdowns: list[str] = []

    for page in ocr_response.pages:
        page_num = page.index + 1
        markdown = page.markdown or ""

        for image_idx, image in enumerate(page.images or [], start=1):
            filename = f"page_{page_num}_image_{image_idx}.md"
            image_path = images_dir / filename
            rel_path = f"images/{filename}"

            if image.image_base64:
                description = describe_image(client, image.image_base64)
                image_path.write_text(description.strip() + "\n", encoding="utf-8")
                print(f"  described {filename}")

            # Replace ![alt](img-id) with a link to the description .md
            markdown = re.sub(
                rf"!\[[^\]]*\]\({re.escape(image.id)}\)",
                f"[Figure {image_idx}]({rel_path})",
                markdown,
            )
            markdown = markdown.replace(f"]({image.id})", f"]({rel_path})")

        for table_idx, table in enumerate(page.tables or [], start=1):
            filename = f"page_{page_num}_table_{table_idx}.md"
            table_path = tables_dir / filename
            table_path.write_text(table.content or "", encoding="utf-8")

            rel_path = f"tables/{filename}"
            markdown = markdown.replace(f"]({table.id})", f"]({rel_path})")
            markdown = re.sub(
                rf"\[{re.escape(table.id)}\]\({re.escape(table.id)}\)",
                f"[Table {table_idx}]({rel_path})",
                markdown,
            )

        page_markdowns.append(markdown.strip())

    full_markdown = "\n\n<!-- page break -->\n\n".join(page_markdowns)
    main_md_path = output_dir / f"{book_id}.md"
    main_md_path.write_text(full_markdown + "\n", encoding="utf-8")

    print(f"Saved {main_md_path}")
    print(f"  tables: {len(list(tables_dir.glob('*.md')))} files in {tables_dir}")
    print(f"  images: {len(list(images_dir.glob('*.md')))} files in {images_dir}")
    return main_md_path

In [ ]:
with PDF_PATH.open("rb") as f:
    pdf_b64 = base64.b64encode(f.read()).decode()

ocr_response = client.ocr.process(
    model="mistral-ocr-latest",
    document={
        "type": "document_url",
        "document_url": f"data:application/pdf;base64,{pdf_b64}",
    },
    include_image_base64=True,
    table_format="markdown", 
)

print(f"OCR complete: {len(ocr_response.pages)} pages")

In [ ]:
save_extraction(ocr_response, OUTPUT_DIR, BOOK_ID, client)